# Connect a Foundry Agent to a custom MCP server and prove the tool call works end-to-end

Your team's inventory data lives in a real internal service but the agents you ship today read a stale CSV the ops team drops on a share drive every Monday morning. The product owner is tired of stale data answers and the security team flagged the share-drive CSV path as a compliance risk. You have one session to stand up a small MCP server that exposes `list_inventory` and `check_stock`, connect a Foundry Agent to it with OAuth passthrough so the user's identity flows through to the server, and prove via the agent trace that both tools fired with the correct typed arguments.

Why MCP and not function calling (Lab 03)? Two different ownership and integration stories:

- **Function calling.** The application is the integration hub. The dispatch loop translates tool-call directives into local Python calls. Good when the tool logic genuinely belongs in the app.
- **MCP.** Tool ownership lives behind a protocol. The agent talks to the MCP server directly via the connector; the application does NOT dispatch. Good when another team owns the tool and ships it independently.

**Time.** Plan for about 55 minutes hands-on. Foundry provisioning is 5 to 8 minutes; the MCP server is a notebook subprocess and starts in seconds; the agent run polls. Budget up to 90 minutes if you debug the MCP transport.

**Cost.** This lab costs under five cents end-to-end. GPT-4o-mini for the agent's reasoning passes plus three MCP tool calls is fractions of a cent. The MCP server hosting is free (notebook process).

This lab maps to AI-103 Domain 2: Implement generative AI and agentic solutions (33% of exam weight), specifically the MCP tool-integration task statement.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus Azure SDKs plus the MCP Python package.
# The MCP spec is still evolving; the lab pins mcp>=1.0,<2.0 and the OAuth
# passthrough behavior was clarified in mcp 1.2.

!pip install --quiet labrun-checks[azure]==0.7.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 "mcp>=1.2,<2.0"

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json
import os
import subprocess
import sys
import time

from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError, ClientAuthenticationError
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.ai.projects import AIProjectClient

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)
from labrun_checks.adapters.azure import AzureCleanupAdapter

LAB_ID = "mcp-tool-integration"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
AGENT_NAME = f"labrun-{LAB_ID}-agent"
MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30

SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
PROJECT_ENDPOINT = None
AGENT_ID = None
THREAD_ID = None
MCP_SERVER_PROCESS = None
MCP_TRANSPORT = None
MCP_URL = None
AZURE_CREDENTIAL = None

PRICE_PER_1M_INPUT_USD = 0.15
PRICE_PER_1M_OUTPUT_USD = 0.60

# Captured run trace for the validators.
RUN_TRACE = None
FINAL_MESSAGE = None

# Deterministic mock data the MCP server returns.
MOCK_INVENTORY = {
    "SKU-A": {"name": "Stainless steel kettle", "stock_level": 42},
    "SKU-B": {"name": "Linen apron, charcoal", "stock_level": 7},
    "SKU-C": {"name": "Bamboo cutting board, large", "stock_level": 19},
}

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. Foundry hosted agents pin {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION, "resource_group": RESOURCE_GROUP}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# Reverse-creation order: local MCP server process first (so its socket
# releases), then the agent, deployment, project, hub, resource group.
# No critical resources; the MCP server is a local-only subprocess and Azure
# resources do not bill hourly at zero traffic.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_process",
        id="mcp-server-subprocess",
        region="local",
        cli_delete_command="(notebook subprocess; terminated via MCP_SERVER_PROCESS.terminate())",
    ),
    CleanupResource(
        type="foundry_agent",
        id=AGENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml agent delete --resource-group {RESOURCE_GROUP} "
            f"--workspace-name {PROJECT_NAME} --agent-id <agent-id-resolved-at-create>"
        ),
        extra={"project_endpoint": PROJECT_ENDPOINT},
    ),
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
        extra={"account_name": AOAI_ACCOUNT_NAME},
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _terminate_mcp_server() -> None:
    global MCP_SERVER_PROCESS
    proc = MCP_SERVER_PROCESS
    if proc is None:
        return
    try:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
    except Exception:
        pass


def _atexit_cleanup() -> None:
    _terminate_mcp_server()
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up a local MCP server with list_inventory and check_stock

Write a small Python file that uses the `mcp` package's reference server implementation. Two tools:

- `list_inventory()` returns the full inventory dict (no parameters).
- `check_stock(sku: str)` returns one item's name and stock level, or an error if the SKU is unknown.

Use the `@server.tool()` decorator. The decorator infers the JSON Schema from your function's type annotations, so the `sku: str` annotation is what produces `inputSchema.properties.sku.type="string"`.

The lab uses stdio transport by default because Colab does not route `127.0.0.1` to outside processes by default. For a self-hosted notebook, HTTP transport on localhost works too. The setup cell detects the environment and configures the right transport.

**Trap to avoid:** the smoke-test client connects to the SAME server you started in this cell. If the server process never came up (an import error, a bad decorator), the smoke test hangs. The cell wraps the smoke test in a five-second timeout and prints a directional error if the handshake never lands.

In [ ]:
# NBVAL_SKIP
# Task 1: Write the MCP server file, start it as a subprocess via stdio
# transport, then smoke-test it with the mcp client SDK.

MCP_SERVER_PATH = "/tmp/labrun_mcp_server.py"
MOCK_INVENTORY_LITERAL = json.dumps(MOCK_INVENTORY, indent=2)

mcp_server_source = f'''
from mcp.server.fastmcp import FastMCP

INVENTORY = {MOCK_INVENTORY_LITERAL}

server = FastMCP("labrun-inventory")


@server.tool()
def list_inventory() -> dict:
    """Return every SKU and its current stock level."""
    return {{"items": [
        {{"sku": sku, **details}} for sku, details in INVENTORY.items()
    ]}}


@server.tool()
def check_stock(sku: str) -> dict:
    """Return the current stock level for a single SKU."""
    if sku not in INVENTORY:
        return {{"error": "sku_not_found", "sku": sku}}
    item = INVENTORY[sku]
    return {{"sku": sku, "name": item["name"], "stock_level": item["stock_level"]}}


if __name__ == "__main__":
    server.run(transport="stdio")
'''

with open(MCP_SERVER_PATH, "w") as f:
    f.write(mcp_server_source)
print(f"MCP server source written to {MCP_SERVER_PATH}")

# Start the server subprocess. stdio transport reads from stdin and writes to
# stdout, so the parent process holds the pipes.
print("Spinning up the MCP server on stdio transport...")
# YOUR CODE: Start the server with subprocess.Popen([sys.executable,
# MCP_SERVER_PATH], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
# stderr=subprocess.PIPE). Store the resulting Popen in MCP_SERVER_PROCESS.

MCP_TRANSPORT = "stdio"
MCP_URL = f"stdio://{MCP_SERVER_PATH}"
print(f"MCP server process pid: {MCP_SERVER_PROCESS.pid}")
print(f"MCP transport: {MCP_TRANSPORT}")
print(f"MCP URL identifier: {MCP_URL}")

# Smoke-test the server via the mcp client SDK. The validator in Checkpoint 1
# uses the same path.
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def _smoke():
    params = StdioServerParameters(command=sys.executable, args=[MCP_SERVER_PATH])
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as cs:
            await cs.initialize()
            return await cs.list_tools()


print("Smoke-testing the MCP server via the mcp client SDK...")
try:
    SMOKE_TOOLS = asyncio.run(asyncio.wait_for(_smoke(), timeout=10))
except asyncio.TimeoutError:
    print("MCP server did not respond within 10 seconds. Check the server source for errors.")
    raise SystemExit(1)

print(f"Smoke test returned {len(SMOKE_TOOLS.tools)} tool(s):")
for t in SMOKE_TOOLS.tools:
    print(f"  - {t.name}: {t.description!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: MCP server lists exactly two tools (list_inventory and
# check_stock) with the expected typed input schemas.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    import asyncio as _asyncio
    from mcp import ClientSession as _CS, StdioServerParameters as _SP
    from mcp.client.stdio import stdio_client as _stdio
    try:
        async def _list():
            params = _SP(command=sys.executable, args=[MCP_SERVER_PATH])
            async with _stdio(params) as (rs, ws):
                async with _CS(rs, ws) as cs:
                    await cs.initialize()
                    return await cs.list_tools()
        tools_result = _asyncio.run(_asyncio.wait_for(_list(), timeout=10))
        tools = {t.name: t for t in tools_result.tools}
        if len(tools) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"MCP server lists {len(tools)} tools, expected exactly 2 "
                    f"(list_inventory and check_stock). Found: {sorted(tools)}"
                ),
            )
        if "list_inventory" not in tools or "check_stock" not in tools:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"MCP server is missing one of the expected tools. Found: {sorted(tools)}. "
                    f"Expected: list_inventory, check_stock."
                ),
            )

        check_stock_schema = tools["check_stock"].inputSchema or {}
        if check_stock_schema.get("type") != "object":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"check_stock inputSchema.type is {check_stock_schema.get('type')!r}, expected 'object'."
                ),
            )
        properties = check_stock_schema.get("properties") or {}
        sku_prop = properties.get("sku") or {}
        if sku_prop.get("type") != "string":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"check_stock inputSchema.properties.sku.type is {sku_prop.get('type')!r}, "
                    f"expected 'string'. Add the `sku: str` type annotation on the function signature."
                ),
            )
        if "sku" not in (check_stock_schema.get("required") or []):
            return CheckpointResult(
                status="fail",
                error_reason="check_stock inputSchema.required is missing 'sku'.",
            )

        list_inv_schema = tools["list_inventory"].inputSchema or {}
        if list_inv_schema.get("type") != "object":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"list_inventory inputSchema.type is {list_inv_schema.get('type')!r}, expected 'object'."
                ),
            )
        if list_inv_schema.get("required") or []:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"list_inventory inputSchema.required is {list_inv_schema.get('required')}, "
                    f"expected empty. list_inventory takes no parameters."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

If the smoke test hangs or the subprocess pid is missing, the server file did not start. Check that `mcp` installed cleanly (the pip cell at the top), and that the server source has the `@server.tool()` decorators with type-annotated function signatures. The decorator infers the JSON Schema from the annotations; without `sku: str` you lose the `type=string` field.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `MCP_SERVER_PROCESS = subprocess.Popen([sys.executable, MCP_SERVER_PATH], stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)`. The Popen returns immediately; the smoke test below verifies it actually started.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
MCP_SERVER_PROCESS = subprocess.Popen(
    [sys.executable, MCP_SERVER_PATH],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
```

</details>

**Wallet check.** Still at $0.00. The MCP server is a local Python subprocess and costs nothing. No tokens fired yet. The hub and project provisioning in the next task is the first thing that touches Azure compute.

## Task 2: Provision Foundry, deploy GPT-4o-mini, and create the agent with the MCP connector

The agent's tool list is one entry of `type="mcp"` (not `"function"`). The MCP connector handles dispatch on the Foundry side; the application does NOT submit tool outputs. This is the big architectural difference vs Lab 03.

MCP connector payload shape:

```python
{
    "type": "mcp",
    "mcp": {
        "server_url": MCP_URL,            # stdio identifier or https URL
        "authentication": {"type": "oauth_passthrough"},
    },
}
```

`oauth_passthrough` is the cert-relevant auth mode: the agent forwards the user's Entra ID token to the MCP server's `Authorization` header on each tool call. The alternative (`static`) uses a fixed credential and is correct for headless service-to-service calls but not for user-context row-level auth.

**Trap to avoid:** declaring the connector tool as `type="function"` and trying to dispatch via `submit_tool_outputs`. The MCP connector handles dispatch on the Foundry side; that path will return 400 because the agent never produces a function-call directive when its tool list is MCP-only.

In [ ]:
# NBVAL_SKIP
# Task 2: Provision Foundry, deploy GPT-4o-mini, create the MCP-connector agent.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME,
    {"location": REGION, "tags": lab_tags, "kind": "Hub",
     "identity": {"type": "SystemAssigned"},
     "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"}},
).result()
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME,
    {"location": REGION, "tags": lab_tags, "kind": "Project",
     "identity": {"type": "SystemAssigned"},
     "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id}},
).result()
PROJECT_ENDPOINT = project_workspace.properties.discovery_url

for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break

deployment_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": MODEL_NAME, "version": MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, deployment_payload,
).result()
print(f"GPT-4o-mini deployment ready: {DEPLOYMENT_NAME}")

# Authenticate against the project and create the MCP-connected agent.
cred = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cred)

mcp_tool = {
    "type": "mcp",
    "mcp": {
        "server_url": MCP_URL,
        "authentication": {"type": "oauth_passthrough"},
    },
}

agent_instructions = (
    "You are an inventory-lookup agent. When the user asks for inventory, call "
    "list_inventory. When the user asks about a specific SKU, call check_stock "
    "with that SKU. Always cite the SKU and stock level you received from the tool. "
    "Never invent inventory numbers."
)

print("Watching the agent decide which MCP tool to call...")
# YOUR CODE: Create the agent with project_client.agents.create_agent(
# model=DEPLOYMENT_NAME, name=AGENT_NAME, instructions=agent_instructions,
# tools=[mcp_tool], metadata={LAB_TAG_KEY: LAB_TAG_VALUE}).
# Store the result in agent.

AGENT_ID = agent.id
print(f"Agent created: {AGENT_NAME} (id: {AGENT_ID})")

for r in CLEANUP_MANIFEST:
    if r.type == "foundry_agent" and "<agent-id-resolved-at-create>" in (r.cli_delete_command or ""):
        r.id = AGENT_ID
        r.cli_delete_command = r.cli_delete_command.replace(
            "<agent-id-resolved-at-create>", AGENT_ID,
        )

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Agent has an MCP connector tool (type=mcp) pointing at the
# local server with authentication.type=oauth_passthrough. Reject
# type=function (Lab 03's pattern) for this lab.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if not AGENT_ID:
            return CheckpointResult(
                status="fail",
                error_reason="AGENT_ID is unset. Re-run Task 2 first.",
            )
        client_local = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
        try:
            agent_now = client_local.agents.get_agent(AGENT_ID)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Agent {AGENT_ID} could not be fetched via client.agents.get_agent.",
            )

        tools = getattr(agent_now, "tools", None) or []
        mcp_tools = []
        function_tools = []
        for t in tools:
            t_dict = t if isinstance(t, dict) else getattr(t, "as_dict", lambda: {})()
            if t_dict.get("type") == "mcp":
                mcp_tools.append(t_dict)
            elif t_dict.get("type") == "function":
                function_tools.append(t_dict)

        if function_tools:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Agent has function-type tools. This lab uses MCP-only tools; the MCP "
                    "connector handles dispatch on the Foundry side. Remove the function tool."
                ),
            )
        if not mcp_tools:
            return CheckpointResult(
                status="fail",
                error_reason="Agent has no MCP-type tools. Pass tools=[mcp_tool] to create_agent.",
            )
        mcp_block = mcp_tools[0].get("mcp", {})
        if mcp_block.get("server_url") != MCP_URL:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"mcp.server_url is {mcp_block.get('server_url')!r}, expected {MCP_URL!r}. "
                    f"Did you pass the stdio URL identifier resolved in Task 1?"
                ),
            )
        auth = mcp_block.get("authentication", {})
        if auth.get("type") != "oauth_passthrough":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"mcp.authentication.type is {auth.get('type')!r}, expected 'oauth_passthrough'. "
                    f"Static credential auth is the wrong mode for this lab."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint says exactly which field is wrong. Common miss: registering a `type=function` tool because Lab 03's pattern is more familiar. This lab is MCP-only; remove any function tool. Second common miss: `authentication.type=static` instead of `oauth_passthrough`.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `project_client.agents.create_agent(model=DEPLOYMENT_NAME, name=AGENT_NAME, instructions=agent_instructions, tools=[mcp_tool], metadata={LAB_TAG_KEY: LAB_TAG_VALUE})`. The `mcp_tool` dict is already built above with the correct shape.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
agent = project_client.agents.create_agent(
    model=DEPLOYMENT_NAME,
    name=AGENT_NAME,
    instructions=agent_instructions,
    tools=[mcp_tool],
    metadata={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

</details>

**Wallet check.** Still at $0.00. Creating the agent is a control-plane call; the MCP connector itself has no per-call fee from Microsoft. Token meter still hasn't started.

## Task 3: Run a single conversation that exercises both MCP tools

Post one user message that nudges the agent into calling BOTH tools: enumerate inventory, then check stock for `SKU-A` and `SKU-B`. The MCP connector dispatches each call directly to your local server; the application does NOT need a manual `submit_tool_outputs` loop here.

Build it in this order:

1. Create a thread.
2. Post the user message that asks for both inventory and stock checks.
3. Start the run and poll until `status` is `completed` or `failed`.
4. Read the final assistant message and store both the run trace and the message for the validator.

In [ ]:
# NBVAL_SKIP
# Task 3: Single conversation turn that nudges the agent into calling both MCP
# tools. With MCP, the connector handles dispatch; the application just polls.

thread = project_client.agents.create_thread()
THREAD_ID = thread.id
print(f"Conversation thread created: {THREAD_ID}")

user_message = (
    "Show me the current inventory, then tell me the stock level for SKU-A and SKU-B "
    "using the inventory tools."
)
project_client.agents.create_message(thread_id=THREAD_ID, role="user", content=user_message)

# YOUR CODE: Start the run with run = project_client.agents.create_run(
# thread_id=THREAD_ID, agent_id=AGENT_ID).

print("Watching the agent talk to the MCP server through the connector...")
delay = 1.0
elapsed = 0.0
ceiling = 60.0
while elapsed < ceiling:
    run = project_client.agents.get_run(thread_id=THREAD_ID, run_id=run.id)
    if run.status in ("completed", "failed", "cancelled", "expired"):
        break
    time.sleep(delay)
    elapsed += delay
    delay = min(delay * 1.5, 5.0)

RUN_TRACE = run
print(f"Run terminal status: {RUN_TRACE.status}")

messages = list(project_client.agents.list_messages(thread_id=THREAD_ID))
for msg in messages:
    if msg.role == "assistant":
        FINAL_MESSAGE = msg.content[0].text.value if msg.content else ""
        break
print()
print("Assistant reply:")
print(FINAL_MESSAGE)

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Run trace shows at least one list_inventory tool call and two
# check_stock calls (one for SKU-A, one for SKU-B), each with correctly typed
# arguments. The run terminated with status=completed.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if RUN_TRACE is None or THREAD_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="RUN_TRACE or THREAD_ID is unset. Re-run Task 3.",
            )
        if RUN_TRACE.status != "completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run status is {RUN_TRACE.status!r}, expected 'completed'. If 'failed', "
                    f"check that the MCP server is still running and reachable."
                ),
            )

        client_local = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
        steps = list(client_local.agents.list_run_steps(thread_id=THREAD_ID, run_id=RUN_TRACE.id))
        tool_calls = []
        for step in steps:
            details = getattr(step, "step_details", None)
            calls = getattr(details, "tool_calls", None) if details else None
            if calls:
                tool_calls.extend(calls)

        # MCP tool calls expose tool name on a 'name' or nested 'mcp' field. Walk both.
        names_with_args = []
        for tc in tool_calls:
            tc_dict = tc if isinstance(tc, dict) else getattr(tc, "as_dict", lambda: {})()
            name = (tc_dict.get("name") or tc_dict.get("mcp", {}).get("name")
                    or getattr(getattr(tc, "function", None), "name", None)
                    or getattr(tc, "name", None))
            raw_args = (tc_dict.get("arguments") or tc_dict.get("mcp", {}).get("arguments")
                        or getattr(getattr(tc, "function", None), "arguments", None)
                        or getattr(tc, "arguments", None) or "{}")
            try:
                args = json.loads(raw_args) if isinstance(raw_args, str) else (raw_args or {})
            except json.JSONDecodeError:
                args = {}
            names_with_args.append((name, args))

        if not any(n == "list_inventory" for n, _ in names_with_args):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run trace has no list_inventory call. Calls seen: {[n for n, _ in names_with_args]}. "
                    f"Tighten the agent instructions to prompt enumeration."
                ),
            )
        check_calls = [args for name, args in names_with_args if name == "check_stock"]
        if len(check_calls) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run trace has {len(check_calls)} check_stock calls, expected at least 2 "
                    f"(one each for SKU-A and SKU-B)."
                ),
            )
        skus_called = {c.get("sku") for c in check_calls}
        if not {"SKU-A", "SKU-B"}.issubset(skus_called):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"check_stock was called with SKUs {sorted(skus_called)}; expected SKU-A and SKU-B."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

If the run never completes, the MCP connector cannot reach the server: confirm `MCP_SERVER_PROCESS.poll() is None` (the subprocess is still alive). If the agent answered without calling either tool, the instructions are not specific enough; the user message in this cell explicitly names both tools, so a model refusal is rare. If only one check_stock call fires, the model batched the SKUs into a single call (a common shortcut); tighten the user message to name them separately.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `run = project_client.agents.create_run(thread_id=THREAD_ID, agent_id=AGENT_ID)`. After that, the polling loop in the cell handles the rest; no `submit_tool_outputs` is needed for MCP-only agents because the connector dispatches on the Foundry side.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
run = project_client.agents.create_run(thread_id=THREAD_ID, agent_id=AGENT_ID)
```

</details>

**Wallet check.** Around three-tenths of a cent. Three tool calls plus the agent's reasoning and final reply is roughly 2,000 input + 400 output tokens, costing about $0.0003 + $0.00024 = $0.00054. The MCP server runs locally at zero infra cost. Coffee remains dominant.

## Task 4: Confirm the agent's final answer cites the mock server's stock-level integers

The most important proof that MCP worked end-to-end: the agent's answer is grounded in the actual server response, not invented from the prompt context. The mock server returns specific integers for `SKU-A` (42) and `SKU-B` (7). If those integers appear in the assistant message, the data flowed from server through connector through model to user.

No new code is needed here. The validator reads `FINAL_MESSAGE` from Task 3 and confirms both integers are present alongside their SKUs.

In [ ]:
# NBVAL_SKIP
# Task 4: No new student code; this cell prints the substrings the validator
# will look for so you can confirm by eye before running the checkpoint.

print("Final assistant message stored in FINAL_MESSAGE.")
print("Validator will confirm both of these appear in the message text:")
for sku, info in MOCK_INVENTORY.items():
    print(f"  - {sku}: stock level {info['stock_level']}")
print()
print("Actual message preview:")
print("-" * 60)
print(FINAL_MESSAGE)
print("-" * 60)

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: FINAL_MESSAGE contains both the SKU-A and SKU-B stock-level
# integers from the mock server, alongside the SKU strings.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not FINAL_MESSAGE:
            return CheckpointResult(
                status="fail",
                error_reason="FINAL_MESSAGE is empty. Did Task 3 run to completion?",
            )
        msg = FINAL_MESSAGE
        msg_lower = msg.lower()

        if "sku-a" not in msg_lower or "sku-b" not in msg_lower:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message does not reference both SKU-A and SKU-B. Got: {msg!r}"
                ),
            )

        sku_a_level = str(MOCK_INVENTORY["SKU-A"]["stock_level"])
        sku_b_level = str(MOCK_INVENTORY["SKU-B"]["stock_level"])
        if sku_a_level not in msg:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message does not contain SKU-A's stock level {sku_a_level!r}. "
                    f"The agent did not ground its answer in the MCP server response."
                ),
            )
        if sku_b_level not in msg:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message does not contain SKU-B's stock level {sku_b_level!r}. "
                    f"The agent did not ground its answer in the MCP server response."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the assistant message names the SKUs but not the integers, the model is paraphrasing the tool response without quoting the stock-level. Tighten the agent instructions to require citing the integer. If neither SKU appears, the model never called the tool successfully (likely a connector reachability problem).

</details>

<details><summary>Hint 2 (direction)</summary>

No code change is needed in this cell; the validator reads `FINAL_MESSAGE` from Task 3. If the validator fails, fix the upstream cell: change the agent instructions to explicitly require citing the integer stock level, then re-run Tasks 3 and 4.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If the agent message lacks the integers, update the instructions in Task 2 to be more explicit:

```python
agent_instructions = (
    "You are an inventory-lookup agent. Call list_inventory first, then "
    "check_stock for each SKU the user names. Your reply MUST cite the SKU "
    "id and the integer stock level the tool returned, for example: 'SKU-A: 42 in stock'. "
    "Never invent inventory numbers."
)
```

Then re-run Tasks 2, 3, and 4.

</details>

**Wallet check.** Still around three-tenths of a cent. Reading the final message is free. Total damage: under one cent.

## Cleanup

Time to tear it all down. The cell terminates the MCP server subprocess first (so its stdio pipes release), then runs the Azure manifest in reverse-creation order. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Terminate the MCP server subprocess and tear down every Azure
# resource in CLEANUP_MANIFEST.

import sys as _sys

_terminate_mcp_server()
print("MCP server subprocess terminated.")

for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

# Patch resource.extra with constants resolved during the lab so the
# Azure adapter sees account_name, service_name, and project_endpoint
# at cleanup time (manifest is built before those constants are set).
for r in CLEANUP_MANIFEST:
    if r.type in ("aoai_deployment", "aoai_rai_policy") and AOAI_ACCOUNT_NAME:
        r.extra = {"account_name": AOAI_ACCOUNT_NAME}
    elif r.type == "search_index" and "SEARCH_SERVICE_NAME" in globals() and SEARCH_SERVICE_NAME:
        r.extra = {"service_name": SEARCH_SERVICE_NAME}
    elif r.type == "foundry_agent" and "PROJECT_ENDPOINT" in globals() and PROJECT_ENDPOINT:
        r.extra = {"project_endpoint": PROJECT_ENDPOINT}

result = run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    _sys.exit(1)

**Session total: under a penny.** Three MCP tool calls plus the agent's reasoning and final reply land in fractions of a cent at GPT-4o-mini rates. The MCP server runs locally at zero infra cost. The Foundry hub, project, agent, and Standard deployment carry no hourly fee at zero traffic. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. Function calling (Lab 03) and MCP integration (this lab) both let the agent invoke external tools. Walk through what changes in your architecture, ownership, and security review when you migrate a tool from a function-call dispatch loop in the application to an MCP server that other agents can also call.

2. Your team is debating whether to expose the MCP server publicly so partner integrators can connect their own agents to your inventory system. What changes about transport, auth, rate limiting, and audit logging compared to the local-process pattern this lab used?

## Exam-Style Practice

**Question 1 (Domain 2, MCP connector authentication):**

You configure a Foundry Agent's MCP connector with `authentication.type=oauth_passthrough` against an internal MCP server that exposes a `get_employee_record(employee_id)` tool. A user signs into the chat app with their Entra ID identity and asks the agent for an employee record. The MCP server is configured to require the caller's identity for row-level authorization. What flows between the components?

A. The agent generates a service principal token, the connector forwards that token to the MCP server, and the server uses the service principal for row-level auth.

B. The user's Entra ID token is forwarded by the connector to the MCP server in the `Authorization: Bearer` header on each tool call; the server validates the token and enforces row-level auth based on the user's identity.

C. The MCP server retrieves the user's identity from the conversation context the agent sends; no token is forwarded.

D. The agent calls the MCP server unauthenticated and the server trusts the agent's IP address as a network-level allowlist.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: `oauth_passthrough` (the configured value) means the USER'S token is forwarded, not a fresh service principal token. The `static` connector type uses a fixed credential but is the wrong mode here.
- B is correct: this is the documented passthrough behavior. The user's Entra ID token is forwarded as a bearer on each tool call. The MCP server validates and authorises against the user's identity. Row-level auth works.
- C is wrong: MCP does not pass conversation context as an authentication signal. The auth is a header on each tool call.
- D is wrong: IP allowlists are not how the MCP connector works. Authentication is an explicit configuration choice, not a network assumption.

</details>

**Question 2 (Domain 2, MCP vs function calling decision):**

Your team owns three internal services exposed as REST APIs. Each service has its own Entra ID auth, its own SDK, and its own deployment cadence. A chat agent needs to call all three. The team is debating between (i) writing three `function`-type tools and dispatching them in the application code, and (ii) wrapping each service as an MCP server and connecting via the MCP connector. The team wants to minimise the application's per-service maintenance load. Which approach fits, and why?

A. Function calling, because MCP servers add a network hop and increase latency by 50 to 200 ms per tool call.

B. MCP, because each service team can own and maintain their MCP server independently; the application's only integration code is the MCP connector configuration, which does not change when a service's internal SDK changes.

C. Function calling, because MCP is still in preview and not production-ready as of GA in March 2026.

D. MCP, because Foundry deprecates function-type tools at GA and only MCP-type tools work after 2026-06-30.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: latency is a real concern but the question asks about maintenance load, not latency. MCP latency is comparable to direct REST calls.
- B is correct: MCP separates tool ownership from application ownership. Each service team ships their own MCP server and changes its internals without breaking the application. This is the AI-103 study guide's explicit framing for MCP tool integration.
- C is wrong: MCP and the MCP connector are GA in Foundry as of March 2026. Both are in scope on the exam.
- D is wrong: function-type tools are NOT deprecated. Both function-type and MCP-type tools are GA and supported. The two patterns are complementary, not sequential.

</details>